In [3]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(path="./chroma_db")

In [5]:
import pandas as pd
import ast
import chromadb
from chromadb.utils import embedding_functions

# ---------- Read CSV file ----------
df = pd.read_csv("../../../data/raw/exercisedb_all_raw_flat.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} records")

# ---------- Convert list-like strings into real lists ----------
def parse_list_field(value):
    """
    Convert strings such as '['back']' or "['shoulders', 'chest']"
    into real Python lists
    """
    if pd.isna(value) or value == "":
        return []
    try:
        # Use ast.literal_eval to convert string -> list
        return ast.literal_eval(value)
    except:
        # Return an empty list if parsing fails
        return []

# ---------- Create documents and metadata for each row ----------
documents = []
metadatas = []
ids = []

for idx, row in df.iterrows():
    # 1. Create exercise_id
    exercise_id = row["exerciseId"]
    
    # 2. Convert list-string fields
    body_parts = parse_list_field(row["bodyParts"])
    equipments = parse_list_field(row["equipments"])
    target_muscles = parse_list_field(row["targetMuscles"])
    secondary_muscles = parse_list_field(row["secondaryMuscles"])
    instructions = parse_list_field(row["instructions"])
    
    # 3. Create document text
    #    Use exercise name plus details
    doc_text = row["name"]
    if instructions:
        doc_text += "\n\n" + "\n".join(instructions)
    
    # 4. Prepare metadata
    #    ChromaDB only accepts str, int, float, and bool metadata values
    #    Therefore, list values must be converted into comma-separated strings
    metadata = {
        "exercise_id": exercise_id,
        "name": row["name"],
        "body_parts": ", ".join(body_parts) if body_parts else "",
        "equipment": ", ".join(equipments) if equipments else "",
        "target_muscles": ", ".join(target_muscles) if target_muscles else "",
        "secondary_muscles": ", ".join(secondary_muscles) if secondary_muscles else "",
        "source": "exercisedb"
    }
    
    # 5. Append to output lists
    documents.append(doc_text)
    metadatas.append(metadata)
    ids.append(exercise_id)

print(f"Prepared {len(documents)} records")

Loaded 1500 records
Prepared 1500 records


In [6]:
# ---------- Create ChromaDB client ----------
client = chromadb.PersistentClient(path="./chroma_db")

# ---------- Define embedding function ----------
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# ---------- Create collection ----------
collection = client.get_or_create_collection(
    name="exercise_instructions",
    embedding_function=embedding_fn,
    metadata={"description": "ExerciseDB 1,500+ instructions with metadata"}
)

# ---------- Split into batches and add documents ----------
BATCH_SIZE = 100

for i in range(0, len(documents), BATCH_SIZE):
    batch_docs = documents[i:i+BATCH_SIZE]
    batch_metas = metadatas[i:i+BATCH_SIZE]
    batch_ids = ids[i:i+BATCH_SIZE]
    
    collection.add(
        documents=batch_docs,
        metadatas=batch_metas,
        ids=batch_ids
    )
    
    print(f"Added {min(i+BATCH_SIZE, len(documents))}/{len(documents)}")

print(f"Import complete! Collection contains {collection.count()} records")

c:\Users\jirap\miniconda3\envs\ML\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jirap\miniconda3\envs\ML\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jirap\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to 

Added 100/1500
Added 200/1500
Added 300/1500
Added 400/1500
Added 500/1500
Added 600/1500
Added 700/1500
Added 800/1500
Added 900/1500
Added 1000/1500
Added 1100/1500
Added 1200/1500
Added 1300/1500
Added 1400/1500
Added 1500/1500
Import complete! Collection contains 1500 records


In [10]:
import re
import chromadb
from chromadb.utils import embedding_functions

# 1. Connect to ChromaDB using the same path
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("exercise_instructions")  # Use the same collection

# 2. Read RAG_CORPUS.md
file_path = "../RAG_CORPUS.md"
with open(file_path, "r", encoding="utf-8") as f:
    content = f.read()

# 3. Use regular expressions to split each rule
#    Format: ### rule_id \n Category: ... \n Applies to: ... \n Rule: ... \n Use in system: ...
pattern = r"### (\w+)\nCategory: (.+?)\nApplies to: (.+?)\nRule: (.+?)\nUse in system: (.+?)(?=\n---\n|\n###|\Z)"

matches = re.findall(pattern, content, re.DOTALL)

print(f"Found {len(matches)} rules")

# 4. Prepare data for ChromaDB
rule_documents = []
rule_metadatas = []
rule_ids = []

for rule_id, category, applies_to, rule_text, use_in_system in matches:
    # Clean text by trimming whitespace
    rule_id = rule_id.strip()
    category = category.strip()
    applies_to = applies_to.strip()
    rule_text = rule_text.strip()
    
    # Create the document text used for search 
    # Use both rule content and conditions to improve search relevance
    doc_content = f"{rule_text} (Condition: {applies_to})"
    
    # Prepare metadata
    # Convert list values to strings because ChromaDB metadata does not accept lists
    metadata = {
        "source": "rule_corpus",
        "category": category,
        "applies_to": applies_to,
        "rule_text": rule_text,  # Store rule text in metadata for later use
        "use_in_system": use_in_system.strip()
    }
    
    rule_documents.append(doc_content)
    rule_metadatas.append(metadata)
    rule_ids.append(rule_id)

# 5. Add documents to ChromaDB in batches for stability
BATCH_SIZE = 50
for i in range(0, len(rule_documents), BATCH_SIZE):
    collection.add(
        documents=rule_documents[i:i+BATCH_SIZE],
        metadatas=rule_metadatas[i:i+BATCH_SIZE],
        ids=rule_ids[i:i+BATCH_SIZE]
    )
    print(f"Added rules {min(i+BATCH_SIZE, len(rule_documents))}/{len(rule_documents)}")

print(f"\nRule Corpus import complete! Collection now contains {collection.count()} records")

Found 80 rules
Added rules 50/80
Added rules 80/80

Rule Corpus import complete! Collection now contains 1580 records


In [11]:
# Check how many rules were inserted
rule_check = collection.get(
    where={"source": "rule_corpus"},
    include=["metadatas"]
)
print(f"Rules in system: {len(rule_check['ids'])}")

# Display the first three rule examples
for i, meta in enumerate(rule_check['metadatas'][:3], 1):
    print(f"{i}. {meta['category']} - {meta['rule_text'][:50]}...")

Rules in system: 80
1. Recovery - If sleep duration is below 6 hours, reduce trainin...
2. Recovery - High stress should reduce workout intensity and pr...
3. Recovery - The same major muscle group should not be trained ...


In [12]:
test_queries = [
    "How should training be adjusted after sleeping less than 6 hours?",
    "Dumbbell chest exercises",
    "How should training change with high blood pressure?",
    "Exercises that use only body weight",
    "Which back exercises should a beginner do?"
]

for q in test_queries:
    print(f"\nQuery: {q}")
    print("-" * 60)
    
    results = collection.query(
        query_texts=[q],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )
    
    for i, (doc_id, doc, meta, dist) in enumerate(zip(
        results['ids'][0],
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        source = meta.get('source', 'unknown')
        name_or_rule = meta.get('name', meta.get('rule_text', ''))[:50]
        
        print(f"  Rank {i} (Score: {1-dist:.4f})")
        print(f"    Source: {source}")
        print(f"    Content: {name_or_rule}...")
        print(f"    Category/Equipment: {meta.get('category', meta.get('equipment', 'N/A'))}")


Query: How should training be adjusted after sleeping less than 6 hours?
------------------------------------------------------------
  Rank 1 (Score: 0.2099)
    Source: exercisedb
    Content: chin-up diagonal...
    Category/Equipment: body weight
  Rank 2 (Score: 0.2003)
    Source: exercisedb
    Content: mixed grip chin-up...
    Category/Equipment: body weight
  Rank 3 (Score: 0.1964)
    Source: exercisedb
    Content: chin-ups (narrow parallel grip)...
    Category/Equipment: body weight
  Rank 4 (Score: 0.1956)
    Source: exercisedb
    Content: chin-up...
    Category/Equipment: body weight
  Rank 5 (Score: 0.1912)
    Source: exercisedb
    Content: one arm chin-up...
    Category/Equipment: body weight

Query: Dumbbell chest exercises
------------------------------------------------------------
  Rank 1 (Score: 0.1598)
    Source: exercisedb
    Content: rounded twisted leg raise (female)...
    Category/Equipment: body weight
  Rank 2 (Score: 0.1590)
    Source: exercis